# Understanding the tradeoffs
- Block Size v/s time to complete 
- Scheduling 

In [24]:
import numpy as np 
import time 

In [25]:
MB = 1*8*(2**12) # Number of random bits for total dataset generation
N = 256 # number of points in the NTT
q = 7681 # prime modulus
W = 3844 # primitive root of unity for NTT modulo q

In [26]:
a = np.random.randint(0, 2, MB)
b = np.random.randint(0, 2, MB)
assert not np.array_equal(a, b), "Input arrays a and b should not be identical."

In [27]:
BLOCK_SIZE = 128 # Size of each block for partitioned convolution

def partitioned_circular_convolution(A, B):

    N = len(A)

    assert len(B) == N, "A and B must have the same length"
    assert N % BLOCK_SIZE == 0, \
        "Input length must be divisible by 128"

    num_blocks = N // BLOCK_SIZE

    # Final circular-convolution output
    result = np.zeros(N, dtype=np.int64)

    for i in range(num_blocks):

        A_block = A[
            i * BLOCK_SIZE:
            (i + 1) * BLOCK_SIZE
        ]

        A_padded = np.zeros(N, dtype=np.int64)

        A_padded[:BLOCK_SIZE] = A_block

        for j in range(num_blocks):

            B_block = B[
                j * BLOCK_SIZE:
                (j + 1) * BLOCK_SIZE
            ]

            B_padded = np.zeros(N, dtype=np.int64)

            B_padded[:BLOCK_SIZE] = B_block

            C_block = np.convolve(
                A_block,
                B_block
            )

            output_start = (i + j) * BLOCK_SIZE

            for k in range(len(C_block)):

                output_index = (
                    output_start + k
                ) % N

                result[output_index] += C_block[k]

    return result

#hw_golden_ref = partitioned_circular_convolution(a, b)

In [28]:
start_time = time.time()
stop_time = time.time()
execution_time = stop_time - start_time

for i in range(6, 12):
    BLOCK_SIZE = 2**i
    start_time = time.time()
    partitioned_circular_convolution(a, b)
    stop_time = time.time()
    execution_time = stop_time - start_time
    print(f"Execution time for block size {BLOCK_SIZE}: {execution_time:.6f} seconds")

Execution time for block size 64: 9.322158 seconds
Execution time for block size 128: 4.425374 seconds
Execution time for block size 256: 2.347519 seconds
Execution time for block size 512: 1.364424 seconds
Execution time for block size 1024: 0.887626 seconds
Execution time for block size 2048: 0.663053 seconds
